# Dynamic Instructions - Context-Aware Agents

## Purpose
Learn how to generate agent instructions dynamically at runtime based on context. This enables personalized agent behavior, user-specific customization, and context-aware responses without creating separate agents for each scenario.

## Key Concepts
- **Dynamic Instructions**: Instructions generated at runtime via function
- **RunContextWrapper**: Type-safe context container passed to instruction builder
- **Agent[ContextType]**: Type parameter specifying expected context structure
- **Context-Aware Behavior**: Agent adapts based on user data, permissions, or state

## Installation

In [ ]:
#!pip install openai
#!pip install openai-agents
#!pip install aws-bedrock-token-generator

## Authentication Setup

In [ ]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from aws_bedrock_token_generator import provide_token

client = AsyncOpenAI(
    api_key=provide_token(),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1",
    project="default"
)

set_default_openai_client(client)
set_default_openai_api("responses")
set_tracing_disabled(True)  # OpenAI-platform tracing can't reach Mantle

## Import Libraries

Import `RunContextWrapper` for context management and `dataclass` for defining context:

In [ ]:
import asyncio
from agents import Agent, Runner, RunContextWrapper
from dataclasses import dataclass

## Step 1: Define Context Model

Create a dataclass or Pydantic model to hold runtime context:

💡 **Design Tip**: Include any data that should influence agent behavior - user info, permissions, preferences, session state, etc.

In [ ]:
@dataclass
class UserContext:
    name: str

## Step 2: Create Instruction Builder Function

Define a function that generates instructions dynamically based on context:

**Function Signature**:
- `ctx: RunContextWrapper[UserContext]` - Context wrapper with user data
- `agent: Agent` - The agent being configured
- Returns: `str` - Generated instructions

🔍 **How It Works**: This function is called automatically when the agent starts, with access to the runtime context.

In [ ]:
def build_instructions(ctx: RunContextWrapper[UserContext], agent: Agent) -> str:
    return f"You are a helpful assistant. Address the user as {ctx.context.name}."

## Step 3: Create Agent with Dynamic Instructions

Pass the builder function as `instructions` and specify the context type:

**Key Details**:
- `Agent[UserContext]` - Type parameter specifies context structure
- `instructions=build_instructions` - Function reference (not string)
- Instructions are generated fresh for each execution

⚡ **Note**: Type safety helps catch errors at development time!

In [ ]:
agent = Agent[UserContext](
    name="Support Assistant",
    instructions=build_instructions,
    model="openai.gpt-5.5",
)

## Step 4: Run with Context

Create context instance and pass to `Runner.run()`:

**Execution Flow**:
1. Create `UserContext` with specific data
2. Pass as `context=` parameter
3. Builder function generates personalized instructions
4. Agent runs with custom instructions

🎯 **Result**: Agent addresses user by name specified in context!

In [ ]:
user_ctx = UserContext(name="John Smith")

result = await Runner.run(
    agent, 
    input = "Name the person you are helping?",
    context=user_ctx)
print(result.final_output)

## 🎉 Congratulations!

You've completed the **Dynamic Instructions** notebook!